# Ejercicio práctico: buscador para una base de conocimiento

Una empresa quiere mejorar el buscador de su portal interno de soporte. La base contiene artículos sobre acceso remoto, facturación, seguridad y dispositivos. Algunas consultas incluyen códigos exactos; otras describen el problema con palabras distintas a las del artículo.

El objetivo es construir una canalización **retrieve-then-rerank** que combine BM25 y embeddings, fusione sus rankings y reordene un conjunto pequeño de candidatos.

Ejecuta las celdas en orden y completa únicamente los bloques marcados con `TODO`.

In [ ]:
import re
from collections import defaultdict

import numpy as np
import pandas as pd
from IPython.display import display
from langchain_core.documents import Document
from rank_bm25 import BM25Okapi
from sentence_transformers import CrossEncoder, SentenceTransformer

pd.set_option("display.max_colwidth", 80)
pd.set_option("display.precision", 4)

In [ ]:
corpus = [
    Document(page_content="Restablecer la contraseña corporativa desde el portal de identidad.", metadata={"id": "KB-101", "categoria": "acceso"}),
    Document(page_content="Configurar la VPN corporativa en macOS para trabajar desde fuera de la oficina.", metadata={"id": "KB-102", "categoria": "acceso"}),
    Document(page_content="Resolver el error 809 de la VPN en Windows revisando el firewall y el protocolo IKEv2.", metadata={"id": "KB-103", "categoria": "acceso"}),
    Document(page_content="Cambiar el teléfono asociado a la autenticación multifactor y registrar un nuevo dispositivo.", metadata={"id": "KB-104", "categoria": "acceso"}),
    Document(page_content="Solicitar la devolución de un cargo duplicado en una factura de suscripción.", metadata={"id": "KB-105", "categoria": "facturacion"}),
    Document(page_content="Solucionar un error 403 al iniciar sesión mediante SSO borrando cookies y renovando la sesión.", metadata={"id": "KB-106", "categoria": "acceso"}),
    Document(page_content="Diagnosticar una batería de portátil que se descarga rápidamente y generar un informe de salud.", metadata={"id": "KB-107", "categoria": "dispositivos"}),
    Document(page_content="Cancelar una suscripción y conservar acceso hasta el final del periodo facturado.", metadata={"id": "KB-108", "categoria": "facturacion"}),
    Document(page_content="Reportar un correo de phishing y preservar las cabeceras para el equipo de seguridad.", metadata={"id": "KB-109", "categoria": "seguridad"}),
    Document(page_content="Corregir problemas de micrófono en videollamadas revisando permisos y dispositivo de entrada.", metadata={"id": "KB-110", "categoria": "dispositivos"}),
]

evaluation_queries = [
    {"id": "Q1", "query": "error 809 al conectar la VPN desde Windows", "relevant": "KB-103"},
    {"id": "Q2", "query": "he cambiado de móvil y ya no recibo códigos", "relevant": "KB-104"},
    {"id": "Q3", "query": "me han cobrado dos veces la misma factura", "relevant": "KB-105"},
    {"id": "Q4", "query": "el navegador devuelve 403 al entrar con SSO", "relevant": "KB-106"},
    {"id": "Q5", "query": "cómo denunciar un correo sospechoso", "relevant": "KB-109"},
    {"id": "Q6", "query": "acceder desde casa a recursos corporativos con un Mac", "relevant": "KB-102"},
]

corpus_df = pd.DataFrame([
    {"id": doc.metadata["id"], "categoria": doc.metadata["categoria"], "texto": doc.page_content}
    for doc in corpus
])
display(corpus_df)

## 1. Recuperadores sparse y dense

Implementa dos funciones que devuelvan un `DataFrame` ordenado de mayor a menor relevancia:

- `rank_bm25(query)`: puntuación BM25 para todos los documentos.
- `rank_dense(query)`: similitud coseno entre el embedding de la consulta y los embeddings documentales.

Cada resultado debe contener `id`, su columna de score y `texto`.

In [ ]:
def tokenize(text):
    return re.findall(r"[a-záéíóúüñ0-9]+(?:-[a-záéíóúüñ0-9]+)?", text.lower())


tokenized_corpus = [tokenize(doc.page_content) for doc in corpus]
bm25 = BM25Okapi(tokenized_corpus)

In [ ]:
def rank_bm25(query):
    # TODO: calcula los scores, ordena los índices y construye el DataFrame.
    raise NotImplementedError

In [ ]:
embedding_model = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)
document_embeddings = embedding_model.encode(
    [doc.page_content for doc in corpus],
    normalize_embeddings=True,
    convert_to_numpy=True,
    show_progress_bar=False,
)

In [ ]:
def rank_dense(query):
    # TODO: codifica la consulta, calcula similitudes y devuelve el ranking.
    raise NotImplementedError

In [ ]:
test_query = evaluation_queries[0]["query"]
display(rank_bm25(test_query).head(5).round(3))
display(rank_dense(test_query).head(5).round(3))

assert rank_bm25(test_query).iloc[0]["id"] == "KB-103"

## 2. Hybrid retrieval

Los scores sparse y dense tienen escalas diferentes. Primero implementa normalización min–max. Después combina ambas señales mediante:

$$s(d)=\alpha\,\hat{s}_{BM25}(d)+(1-\alpha)\,\hat{s}_{dense}(d)$$

Utiliza `alpha=0.5` como valor inicial. Finalmente implementa RRF para fusionar las posiciones sin depender de los scores.

In [ ]:
def minmax_normalize(values):
    # TODO: devuelve valores entre 0 y 1. Si todos son iguales, devuelve ceros.
    raise NotImplementedError


sample = np.array([2.0, 4.0, 6.0])
assert np.allclose(minmax_normalize(sample), [0.0, 0.5, 1.0])

In [ ]:
def score_fusion(query, alpha=0.5):
    # TODO: combina los rankings por id, normaliza ambos scores y calcula score_hybrid.
    raise NotImplementedError

In [ ]:
def reciprocal_rank_fusion(rankings, k=60):
    # TODO: acumula 1 / (k + posición) para cada documento.
    raise NotImplementedError


def rrf_fusion(query, k=60):
    sparse_ids = rank_bm25(query)["id"].tolist()
    dense_ids = rank_dense(query)["id"].tolist()
    scores = reciprocal_rank_fusion([sparse_ids, dense_ids], k=k)
    result = pd.DataFrame(scores, columns=["id", "score_rrf"])
    return result.merge(corpus_df[["id", "texto"]], on="id")

In [ ]:
query = evaluation_queries[1]["query"]
comparison = pd.DataFrame({
    "posición": range(1, 6),
    "BM25": rank_bm25(query).head(5)["id"].tolist(),
    "dense": rank_dense(query).head(5)["id"].tolist(),
    "score fusion": score_fusion(query).head(5)["id"].tolist(),
    "RRF": rrf_fusion(query).head(5)["id"].tolist(),
})
display(comparison)

## 3. Evaluación

Se utilizarán dos métricas simples:

- **Hit@k:** proporción de consultas cuyo documento relevante aparece entre los primeros `k` resultados.
- **MRR@k:** promedio del inverso de la posición del primer documento relevante, con valor cero si no aparece en el Top-k.

Ejecuta la comparación y observa si el híbrido mejora todas las consultas o si el resultado depende del tipo de consulta.

In [ ]:
def evaluate_ranker(ranker, queries, k=3):
    reciprocal_ranks = []
    hits = []
    for item in queries:
        ranking = ranker(item["query"])["id"].head(k).tolist()
        if item["relevant"] in ranking:
            position = ranking.index(item["relevant"]) + 1
            hits.append(1)
            reciprocal_ranks.append(1 / position)
        else:
            hits.append(0)
            reciprocal_ranks.append(0)
    return {"Hit@3": np.mean(hits), "MRR@3": np.mean(reciprocal_ranks)}


evaluation = pd.DataFrame({
    "BM25": evaluate_ranker(rank_bm25, evaluation_queries),
    "dense": evaluate_ranker(rank_dense, evaluation_queries),
    "score fusion": evaluate_ranker(score_fusion, evaluation_queries),
    "RRF": evaluate_ranker(rrf_fusion, evaluation_queries),
}).T
display(evaluation.round(3))

## 4. Reranking de candidatos

Recupera seis candidatos con RRF y reordénalos con un cross-encoder. La función debe conservar exactamente el mismo conjunto de candidatos y añadir `score_reranker` y `posición_reranker`.

El modelo mejora el orden de los candidatos; no puede recuperar un artículo que no haya llegado al Top-k inicial.

In [ ]:
reranker = CrossEncoder("cross-encoder/mmarco-mMiniLMv2-L12-H384-v1")


def rerank(query, candidates):
    # TODO: crea pares (query, texto), predice los scores y ordena el resultado.
    raise NotImplementedError

In [ ]:
query = evaluation_queries[4]["query"]
candidates = rrf_fusion(query).head(6).copy()
candidates["posición_inicial"] = range(1, len(candidates) + 1)
reranked = rerank(query, candidates)

assert set(candidates["id"]) == set(reranked["id"])
display(reranked[["posición_reranker", "id", "score_reranker", "texto"]].round(3))

## 5. Reto breve: MaxSim

Cada fila de `query_vectors` representa un token de consulta y cada fila de `document_vectors` un token documental. Implementa:

$$\operatorname{MaxSim}(q,d)=\sum_i\max_j(q_i\cdot d_j)$$

Los vectores ya están normalizados, por lo que el producto escalar equivale a la similitud coseno.

In [ ]:
query_vectors = np.array([
    [1.0, 0.0, 0.0],
    [0.0, 1.0, 0.0],
])
document_vectors = np.array([
    [0.8, 0.6, 0.0],
    [0.6, 0.8, 0.0],
    [0.0, 0.0, 1.0],
])


def maxsim(query_vectors, document_vectors):
    # TODO: construye la matriz de similitud, toma el máximo por token de consulta y suma.
    raise NotImplementedError


assert np.isclose(maxsim(query_vectors, document_vectors), 1.6)

## Preguntas finales

1. ¿Qué tipo de consultas favorece a BM25 en este corpus? ¿Cuáles favorecen al recuperador dense?
2. ¿Qué se pierde al utilizar RRF en lugar de fusionar scores calibrados?
3. ¿Por qué se aplica el cross-encoder únicamente sobre un Top-k?
4. Si los artículos incluyeran capturas de pantalla, ¿qué codificadores deberían producir representaciones compatibles para permitir búsqueda texto → imagen?